# Appendix 5. pandas와 Matplotlib 연결

## 1. Goal

pandas의 `Series.plot`과 `DataFrame.plot`은 Matplotlib을 감싼 편의 API입니다. 빠르게 chart를 만들 수 있지만, 반환된 `Axes`를 이해해야 제목·축·범위·legend·subplot layout을 일관되게 다룰 수 있습니다.

이 notebook은 pandas로 집계한 표를 chart로 만들고, Appendix 4에서 배운 Figure와 Axes API로 후처리하는 연결 과정을 연습합니다. line, bar, histogram, box, scatter, hexbin처럼 질문에 맞는 chart를 고르고, 정렬·집계·결측 처리·subplot·style·annotation까지 하나의 흐름으로 다룹니다.

## 2. Setup

프로젝트 lock의 pandas 버전은 2.3.3이고 Matplotlib 버전은 3.11.0입니다.

모든 값은 고정된 합성 ICU 데이터입니다. chart 모양과 API 반환값을 이해하기 위한 자료이며, target별 차이를 임상적 결론이나 feature 선택 근거로 해석하지 않습니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.axes import Axes
from matplotlib.figure import Figure


## 3. Steps

각 `###` 절은 pandas plotting의 활용 개념군을, `####` 절은 표에서 chart로 이어지는 하나의 실행 단위를 나타냅니다. 반환 객체를 확인한 뒤 chart 선택, 다중 Axes, 집계, 후처리 순서로 진행합니다.

### 3-1. pandas plotting 반환 객체

#### 3-1-1. Series.plot이 생성한 Axes 받기

`Series.plot.line()`을 호출하면 pandas가 Figure와 Axes를 만들고 데이터를 그린 뒤 `Axes`를 반환합니다. 반환된 Axes의 `get_figure()`로 상위 Figure에 접근할 수 있습니다. 즉, pandas로 시작해도 이후 단계는 일반 Matplotlib 객체 API와 같습니다.

In [ ]:
observation_times = pd.date_range(
    "2026-01-01 00:00", periods=8, freq="30min", tz="Asia/Seoul"
)
heart_rate_over_time = pd.Series(
    [78, 81, 86, 84, 88, 90, 87, 85],
    index=observation_times,
    name="heart_rate",
)
returned_ax = heart_rate_over_time.plot.line(
    figsize=(7, 3.5),
    marker="o",
    title="Heart rate over time",
)
series_figure = returned_ax.get_figure()
print(
    {
        "returned_type": type(returned_ax).__name__,
        "figure_type": type(series_figure).__name__,
        "line_count": len(returned_ax.lines),
    }
)


#### 3-1-2. 미리 만든 Axes를 pandas plot에 전달하기

layout을 직접 통제하고 싶다면 Matplotlib에서 Figure와 Axes를 먼저 만들고 pandas plot의 `ax=` 인자로 전달합니다. pandas는 전달받은 Axes에 chart를 그리고 **같은 Axes 객체를 반환**합니다. 이 패턴은 여러 요약표를 하나의 Figure에 배치할 때 가장 예측하기 쉽습니다.

In [ ]:
patient_features = pd.DataFrame(
    {
        "patient_id": ["P101", "P102", "P103", "P104", "P105", "P106", "P107", "P108"],
        "target": [0, 0, 0, 0, 1, 1, 1, 1],
        "heart_rate_mean": [78, 82, 85, 88, 90, 94, 87, 91],
        "lactate_mean": [1.4, 1.8, 2.1, 1.7, 2.8, 3.2, 2.5, 3.0],
        "lactate__missing": [0, 1, 0, 0, 1, 1, 0, 1],
        "fio2__missing": [0, 0, 1, 0, 1, 1, 1, 0],
    }
).set_index("patient_id")
missing_rates = patient_features[["lactate__missing", "fio2__missing"]].mean()

prepared_figure, prepared_axes = plt.subplots(
    1, 2, figsize=(10, 3.5), layout="constrained"
)
bar_ax = missing_rates.plot.bar(ax=prepared_axes[0], color="tab:blue")
hist_ax = patient_features["heart_rate_mean"].plot.hist(
    ax=prepared_axes[1], bins=4, color="tab:orange", edgecolor="white"
)
prepared_figure.suptitle("Pandas plots drawn on prepared Matplotlib Axes")
print({"bar_identity": bar_ax is prepared_axes[0], "hist_identity": hist_ax is prepared_axes[1]})


### 3-2. 데이터 질문과 chart API

#### 3-2-1. line, bar, hist, scatter 선택 기준

시간 순서의 변화는 `plot.line`, 범주별 크기 비교는 `plot.bar`, 한 수치의 분포는 `plot.hist`, 두 수치 관계는 `plot.scatter`가 기본 선택입니다. chart를 고르기 전에 index와 column이 각각 어떤 분석 단위를 표현하는지 확인합니다.

In [ ]:
scatter_figure, scatter_axes = plt.subplots(
    figsize=(6, 4), layout="constrained"
)
scatter_ax = patient_features.plot.scatter(
    x="heart_rate_mean",
    y="lactate_mean",
    c="target",
    colormap="coolwarm",
    ax=scatter_axes,
)
scatter_ax.set_title("Heart rate and lactate by target")


#### 3-2-2. box plot으로 group별 분포 비교하기

box plot은 중앙값, 사분위 범위와 극단값 후보를 한 번에 비교할 때 유용합니다. pandas의 `DataFrame.boxplot`은 group column을 `by`로 지정할 수 있습니다. box plot만으로 분포 모양이나 sample size를 모두 알 수 없으므로 group별 count도 함께 확인합니다.

In [ ]:
box_figure, box_axes = plt.subplots(
    figsize=(6, 4),
    layout="constrained",
)
box_ax = (
    patient_features.reset_index()
    .boxplot(
        column="lactate_mean",
        by="target",
        ax=box_axes,
        grid=False,
    )
)
box_figure.suptitle("")
box_ax.set(
    title="Lactate distribution by target",
    xlabel="Target",
    ylabel="Mean lactate [mmol/L]",
)
target_sample_size = patient_features.groupby("target").size()
target_sample_size


#### 3-2-3. bar와 barh에서 category 순서 정하기

bar chart의 category 순서는 pandas 객체의 Index 순서를 따릅니다. 값의 크기를 비교하려면 먼저 `sort_values`로 정렬하고, label이 길면 `barh`를 사용합니다. 시간 순서나 domain 순서가 중요한 category는 값 크기로 임의 정렬하지 않습니다.

In [ ]:
target_counts = (
    patient_features["target"]
    .value_counts()
    .sort_index()
    .rename(index={0: "negative", 1: "positive"})
)
count_figure, count_axes = plt.subplots(
    figsize=(6, 3),
    layout="constrained",
)
count_ax = target_counts.plot.barh(
    ax=count_axes,
    color=["tab:blue", "tab:orange"],
)
count_ax.set(
    title="Patient count by target",
    xlabel="Patient count",
    ylabel="Target",
)


#### 3-2-4. hexbin으로 겹치는 point의 밀도 보기

scatter plot은 point가 많거나 같은 위치에 겹치면 밀도를 읽기 어렵습니다. `plot.hexbin`은 x·y 공간을 육각형 bin으로 나누고 각 bin의 관측 수를 색으로 표시합니다. `gridsize`가 결론을 바꿀 수 있으므로 여러 값을 비교하고 sample size가 작은 데이터에서는 과도하게 해석하지 않습니다.

In [ ]:
hexbin_figure, hexbin_axes = plt.subplots(
    figsize=(6, 4),
    layout="constrained",
)
hexbin_ax = patient_features.plot.hexbin(
    x="heart_rate_mean",
    y="lactate_mean",
    gridsize=4,
    mincnt=1,
    cmap="viridis",
    ax=hexbin_axes,
)
hexbin_ax.set_title("Patient density in feature space")


### 3-3. 여러 Axes 다루기

#### 3-3-1. DataFrame.plot subplots 반환값 다루기

`subplots=True`는 선택한 각 column을 별도 Axes에 그립니다. 이때 반환값은 하나의 Axes가 아니라 Axes 배열입니다. `layout=(rows, columns)`으로 모양을 정하고, 반환 배열을 `ravel()`로 순회하면 각 Axes에 공통 설정을 적용할 수 있습니다.

서로 단위가 다른 column을 같은 y축에 겹치기보다 별도 subplot으로 나누면 축 의미가 명확해집니다.

In [ ]:
subplot_axes = patient_features[["heart_rate_mean", "lactate_mean"]].plot(
    subplots=True,
    layout=(1, 2),
    figsize=(10, 3.5),
    marker="o",
    sharex=True,
    legend=False,
)
for subplot_ax, title, ylabel in zip(
    subplot_axes.ravel(),
    ["Mean heart rate", "Mean lactate"],
    ["BPM", "mmol/L"],
    strict=True,
):
    subplot_ax.set(title=title, xlabel="Patient", ylabel=ylabel)
    subplot_ax.grid(axis="y", alpha=0.25)
subplot_axes.ravel()[0].get_figure().suptitle("Separate units on separate Axes")


#### 3-3-2. Figure와 Axes grid를 먼저 설계하기

여러 질문을 한 Figure에 배치할 때는 Matplotlib으로 Axes grid를 먼저 만든 뒤 각 pandas plot에 `ax=`로 전달합니다. 이렇게 하면 각 chart가 어느 위치에 그려지는지 명확하고, Figure 크기와 layout도 한곳에서 관리할 수 있습니다.

In [ ]:
dashboard_figure, dashboard_axes = plt.subplots(
    2,
    2,
    figsize=(10, 7),
    layout="constrained",
    squeeze=False,
)
patient_features["heart_rate_mean"].plot.hist(
    bins=4,
    ax=dashboard_axes[0, 0],
    title="Heart rate distribution",
)
patient_features["lactate_mean"].plot.hist(
    bins=4,
    ax=dashboard_axes[0, 1],
    title="Lactate distribution",
)
missing_rates.sort_values().plot.barh(
    ax=dashboard_axes[1, 0],
    title="Missing indicator rate",
)
patient_features.plot.scatter(
    x="heart_rate_mean",
    y="lactate_mean",
    c="target",
    colormap="coolwarm",
    ax=dashboard_axes[1, 1],
    title="Feature relationship",
)
dashboard_figure.suptitle("EDA question grid")


#### 3-3-3. 같은 scale을 비교할 때 axis 공유하기

동일한 변수의 group별 분포처럼 scale이 같은 chart는 `sharex` 또는 `sharey`로 축을 공유하면 비교가 쉬워집니다. 서로 다른 단위나 범위를 억지로 공유하면 차이가 왜곡되므로, 공유할 변수와 단위를 먼저 확인합니다.

In [ ]:
comparison_figure, comparison_axes = plt.subplots(
    1,
    2,
    figsize=(9, 3.5),
    sharex=True,
    sharey=True,
    layout="constrained",
)
for target_value, target_ax in zip(
    [0, 1],
    comparison_axes,
    strict=True,
):
    patient_features.loc[
        patient_features["target"].eq(target_value),
        "heart_rate_mean",
    ].plot.hist(
        bins=[75, 80, 85, 90, 95],
        ax=target_ax,
        alpha=0.75,
    )
    target_ax.set(
        title=f"Target {target_value}",
        xlabel="Mean heart rate [BPM]",
        ylabel="Patient count",
    )
comparison_figure.suptitle("Shared scale for group comparison")


### 3-4. 집계 결과 시각화

#### 3-4-1. GroupBy 결과를 표로 확인한 뒤 시각화하기

EDA에서는 raw 행을 바로 plot하기보다 먼저 분석 grain에 맞게 집계합니다. 아래 예제는 target별 missing indicator 평균을 계산합니다. 0/1 indicator의 평균은 비율이므로, 집계표의 행은 feature, 열은 target, 값은 missing rate가 됩니다. 표를 먼저 확인하고 그 동일한 객체를 chart에 전달합니다.

In [ ]:
missing_columns = [column for column in patient_features if column.endswith("__missing")]
missingness_by_target = patient_features.groupby("target")[missing_columns].mean().T
missingness_by_target["absolute_gap"] = (
    missingness_by_target[1] - missingness_by_target[0]
).abs()
missingness_by_target.sort_values("absolute_gap", ascending=False)


In [ ]:
group_figure, group_axes = plt.subplots(
    figsize=(7, 4), layout="constrained"
)
group_ax = missingness_by_target[[0, 1]].plot.bar(ax=group_axes)
group_ax.set(
    title="Missing indicator rate by target",
    xlabel="Feature",
    ylabel="Missing rate",
    ylim=(0, 1),
)
group_ax.legend(title="Target")
group_ax.tick_params(axis="x", labelrotation=20)
group_ax.grid(axis="y", alpha=0.25)


#### 3-4-2. DatetimeIndex, resample과 시간 chart

시간 chart는 x축 label이 `DatetimeIndex`일 때 pandas와 Matplotlib의 날짜 locator·formatter를 활용할 수 있습니다. 먼저 `resample`로 시간당 측정 수를 집계한 뒤 line chart를 그립니다. 여기서 y축은 patient 수가 아니라 해당 시간 bucket의 측정 행 수입니다.

In [ ]:
event_index = pd.to_datetime(
    [
        "2026-01-01 00:05+09:00",
        "2026-01-01 00:35+09:00",
        "2026-01-01 01:10+09:00",
        "2026-01-01 01:45+09:00",
        "2026-01-01 02:15+09:00",
        "2026-01-01 02:40+09:00",
        "2026-01-01 03:20+09:00",
    ]
)
measurement_events = pd.Series(1, index=event_index, name="measurement_count").sort_index()
measurements_by_hour = measurement_events.resample("1h").sum()
time_figure, time_axes = plt.subplots(
    figsize=(7, 3.5), layout="constrained"
)
time_ax = measurements_by_hour.plot.line(ax=time_axes, marker="o")
time_ax.set(title="Measurements by hour", xlabel="Observation hour", ylabel="Row count")
time_ax.grid(axis="y", alpha=0.25)


#### 3-4-3. value_counts 결과를 정렬해 그리기

범주 빈도는 `value_counts`로 집계한 Series를 먼저 확인한 뒤 chart로 만듭니다. 기본값은 빈도순이므로 domain 순서가 필요하면 `sort_index`나 명시적인 category 순서를 적용합니다. chart title과 axis label에는 count인지 rate인지 분명히 적습니다.

In [ ]:
missing_count_per_patient = patient_features[
    ["lactate__missing", "fio2__missing"]
].sum(axis=1)
missing_count_distribution = (
    missing_count_per_patient.value_counts()
    .sort_index()
    .rename_axis("missing_feature_count")
)
missing_count_figure, missing_count_axes = plt.subplots(
    figsize=(6, 3.5),
    layout="constrained",
)
missing_count_ax = missing_count_distribution.plot.bar(
    ax=missing_count_axes,
    color="tab:purple",
)
missing_count_ax.set(
    title="Patients by number of missing features",
    xlabel="Missing feature count",
    ylabel="Patient count",
)


#### 3-4-4. crosstab 비율을 stacked bar로 비교하기

두 범주의 관계는 `crosstab`으로 count 또는 비율 table을 만든 뒤 stacked bar로 비교할 수 있습니다. `normalize="index"`를 사용하면 각 row의 합이 1이므로 y축 단위는 count가 아니라 rate입니다. 반드시 원래 sample count도 함께 확인합니다.

In [ ]:
heart_rate_band = pd.cut(
    patient_features["heart_rate_mean"],
    bins=[0, 84, 89, 200],
    labels=["under_85", "85_to_89", "90_plus"],
)
target_rate_by_band = pd.crosstab(
    heart_rate_band,
    patient_features["target"],
    normalize="index",
)
band_sample_size = pd.crosstab(
    heart_rate_band,
    patient_features["target"],
)
crosstab_figure, crosstab_axes = plt.subplots(
    figsize=(7, 4),
    layout="constrained",
)
crosstab_ax = target_rate_by_band.plot.bar(
    stacked=True,
    ylim=(0, 1),
    ax=crosstab_axes,
    colormap="coolwarm",
)
crosstab_ax.set(
    title="Target composition by heart-rate band",
    xlabel="Mean heart-rate band",
    ylabel="Target rate",
)
crosstab_ax.legend(title="Target")
band_sample_size


#### 3-4-5. rolling 결과와 원본 시계열 함께 그리기

시간 chart에서는 raw Series와 rolling summary를 함께 그리면 단기 변동과 완만한 흐름을 구분할 수 있습니다. rolling window의 단위와 `min_periods`를 명시하고, smoothed line을 원본 관측값처럼 해석하지 않습니다.

In [ ]:
heart_rate_rolling_mean = heart_rate_over_time.rolling(
    window=3,
    min_periods=1,
).mean()
rolling_figure, rolling_axes = plt.subplots(
    figsize=(7, 3.5),
    layout="constrained",
)
heart_rate_over_time.plot.line(
    ax=rolling_axes,
    marker="o",
    alpha=0.45,
    label="Observed",
)
heart_rate_rolling_mean.plot.line(
    ax=rolling_axes,
    linewidth=2.5,
    label="Rolling mean (3 observations)",
)
rolling_axes.set(
    title="Observed and rolling heart rate",
    xlabel="Observed at",
    ylabel="Heart rate [BPM]",
)
rolling_axes.legend()


### 3-5. Axes 후처리

#### 3-5-1. 반환된 Axes에 annotation과 기준선 추가하기

pandas plot이 끝난 뒤에도 반환 Axes에 Matplotlib API를 계속 적용할 수 있습니다. `axhline`은 품질 기준선처럼 y값이 일정한 선을 추가하고, `annotate`는 특정 관측값을 설명합니다. 기준선은 반드시 label과 근거를 함께 제공해야 하며, 탐색 중 임의로 만든 값을 정책 threshold처럼 표현하지 않습니다.

In [ ]:
returned_ax.set(xlabel="Observed at", ylabel="Heart rate [BPM]")
returned_ax.grid(axis="y", alpha=0.25)
reference_line = returned_ax.axhline(90, color="tab:red", linestyle="--", label="Example reference")
returned_ax.annotate(
    "Peak",
    xy=(heart_rate_over_time.idxmax(), heart_rate_over_time.max()),
    xytext=(0, 12),
    textcoords="offset points",
    ha="center",
)
returned_ax.legend()
returned_ax.get_figure().autofmt_xdate()


#### 3-5-2. Secondary y-axis는 단위가 다를 때 제한적으로 사용하기

`secondary_y`는 서로 다른 단위의 두 Series를 같은 x축에 겹칠 때 보조 y축을 만듭니다. 반환값은 왼쪽 primary Axes이고 오른쪽 Axes는 `right_ax`로 접근할 수 있습니다. 두 축의 scale 차이가 관계를 과장할 수 있으므로, 비교 목적이 분명하지 않으면 별도 subplot을 우선합니다.

In [ ]:
operational_summary = pd.DataFrame(
    {
        "positive_rate": [0.12, 0.15, 0.14, 0.18],
        "latency_ms": [82, 95, 88, 110],
    },
    index=pd.date_range("2026-01-01", periods=4, freq="h"),
)
primary_ax = operational_summary.plot.line(
    y=["positive_rate", "latency_ms"],
    secondary_y=["latency_ms"],
    figsize=(7, 3.5),
    marker="o",
)
right_ax = primary_ax.right_ax
primary_ax.set_ylabel("Positive rate")
right_ax.set_ylabel("Latency [ms]")
primary_ax.set_title("Two units with an explicit secondary axis")


#### 3-5-3. Rate axis와 tick label 형식 맞추기

비율 chart의 내부 값이 0에서 1 사이라면 y축도 같은 범위로 고정하고 tick label을 percentage로 표시할 수 있습니다. 형식만 percentage로 바꾼다고 count가 rate가 되는 것은 아니므로 source table의 분모와 normalize 방식을 먼저 확인합니다.

In [ ]:
crosstab_ax.set_ylim(0, 1)
crosstab_ax.yaxis.set_major_formatter("{x:.0%}")
crosstab_ax.tick_params(axis="x", labelrotation=0)
crosstab_ax.grid(axis="y", alpha=0.2)


#### 3-5-4. 반환된 Artist의 style 세부 조정하기

pandas plot이 만든 Axes에는 Matplotlib Artist가 들어 있습니다. 반환된 Axes의 `lines`, `patches`, legend를 이용하면 특정 선이나 막대의 색·두께·투명도를 조정할 수 있습니다. style은 의미를 구분하는 데 사용하고 장식만 늘리지 않습니다.

In [ ]:
primary_line = primary_ax.lines[0]
secondary_line = right_ax.lines[0]
primary_line.set(
    color="tab:blue",
    linewidth=2.5,
    marker="o",
)
secondary_line.set(
    color="tab:orange",
    linewidth=2.0,
    linestyle="--",
    marker="s",
)
primary_ax.grid(axis="y", alpha=0.2)


### 3-6. Chart 입력값과 해석 범위 점검

#### 3-6-1. 정렬과 top-N을 table에서 먼저 결정하기

chart 안에서 category 순서를 임의로 바꾸지 말고 source Series나 DataFrame을 먼저 정렬합니다. 항목이 많아 top-N만 표시한다면 제외된 항목 수와 선택 기준을 함께 기록합니다. 아래 예제는 모든 missing indicator를 rate 기준으로 정렬합니다.

In [ ]:
ranked_missing_rates = missing_rates.sort_values(
    ascending=True
)
ranking_figure, ranking_axes = plt.subplots(
    figsize=(6, 3.5),
    layout="constrained",
)
ranking_ax = ranked_missing_rates.plot.barh(
    ax=ranking_axes,
    color="tab:green",
)
ranking_ax.set(
    title="Missing indicators ranked by rate",
    xlabel="Missing rate",
    ylabel="Feature",
    xlim=(0, 1),
)
ranking_ax.xaxis.set_major_formatter("{x:.0%}")
for patch, value in zip(
    ranking_ax.patches,
    ranked_missing_rates,
    strict=True,
):
    ranking_ax.annotate(
        f"{value:.0%}",
        xy=(value, patch.get_y() + patch.get_height() / 2),
        xytext=(4, 0),
        textcoords="offset points",
        va="center",
    )


#### 3-6-2. 결측값을 숨기지 않고 chart에서 확인하기

line chart는 결측 시점에서 선이 끊길 수 있습니다. chart를 그리기 위해 임의로 결측을 채우지 말고, 먼저 원본 Series의 결측 위치를 확인합니다. 보간이나 대체가 필요하다면 원본과 구분되는 Series를 만들고 처리 규칙을 명시합니다.

In [ ]:
heart_rate_with_gap = heart_rate_over_time.astype("float64").copy()
heart_rate_with_gap.iloc[3] = float("nan")
gap_figure, gap_axes = plt.subplots(
    figsize=(7, 3.5),
    layout="constrained",
)
gap_ax = heart_rate_with_gap.plot.line(
    ax=gap_axes,
    marker="o",
    label="Observed with gap",
)
heart_rate_with_gap.dropna().plot(
    ax=gap_axes,
    color="tab:blue",
    linestyle="none",
    marker="o",
    label="Available observations",
)
gap_ax.set(
    title="Missing observation remains visible as a gap",
    xlabel="Observed at",
    ylabel="Heart rate [BPM]",
)
gap_ax.legend()


## 4. Checks

pandas plot의 반환 타입, 전달한 Axes identity, subplot 배열, 시간 집계와 secondary axis를 확인합니다. chart 값이 이상하면 시각화 설정 전에 source Series·DataFrame의 index, shape, 집계 grain을 먼저 검사하세요.

In [ ]:
assert isinstance(returned_ax, Axes)
assert isinstance(series_figure, Figure)
assert returned_ax.figure is series_figure
assert bar_ax is prepared_axes[0]
assert hist_ax is prepared_axes[1]
assert scatter_ax is scatter_axes
assert box_ax is box_axes
assert int(target_sample_size.sum()) == len(patient_features)
assert count_ax is count_axes
assert hexbin_ax is hexbin_axes
assert subplot_axes.size == 2
assert dashboard_axes.shape == (2, 2)
assert comparison_axes.size == 2
assert group_ax is group_axes
assert int(missing_count_distribution.sum()) == len(patient_features)
assert band_sample_size.to_numpy().sum() == len(patient_features)
assert target_rate_by_band.sum(axis=1).round(8).eq(1.0).all()
assert len(heart_rate_rolling_mean) == len(heart_rate_over_time)
assert measurements_by_hour.sum() == len(measurement_events)
assert time_ax is time_axes
assert reference_line in returned_ax.lines
assert isinstance(primary_ax, Axes)
assert isinstance(right_ax, Axes)
assert right_ax is not primary_ax
assert primary_line in primary_ax.lines
assert secondary_line in right_ax.lines
assert ranking_ax is ranking_axes
assert ranked_missing_rates.index.tolist() == ["lactate__missing", "fio2__missing"]
assert int(heart_rate_with_gap.isna().sum()) == 1
assert gap_ax is gap_axes
print("All appendix checks passed.")
plt.close("all")


## 5. Next Steps

- pandas로 먼저 분석 grain이 분명한 Series 또는 DataFrame을 만들고 count, rate, 분모와 정렬 순서를 확인한 뒤 plot합니다.
- 시간 변화는 line, 범주 크기는 bar·barh, 분포 비교는 hist·box, point 밀도는 scatter·hexbin을 우선 검토합니다.
- 여러 chart를 비교할 때는 동일한 변수와 단위에만 axis를 공유하고, rate axis는 0에서 1 범위와 percentage 형식을 함께 적용합니다.
- 결측값을 chart 때문에 임의로 채우지 말고 원본의 gap을 먼저 확인한 뒤 별도의 처리 Series를 만듭니다.
- layout 통제가 필요하면 Matplotlib에서 Figure와 Axes를 먼저 만들고 pandas plot에 `ax=`로 전달합니다.
- pandas plot이 반환한 Axes에는 Matplotlib의 title, label, limit, grid, annotation, legend API를 그대로 사용합니다.
- 단위가 다른 값은 별도 subplot을 우선하고, secondary axis는 비교 목적과 두 축 단위를 명시할 때만 사용합니다.
- ch01 EDA에서는 같은 패턴을 versioned PhysioNet artifact에 적용하되, chart를 feature 선택이나 sealed-test tuning 근거로 사용하지 않습니다.

## 6. References

이 notebook의 설명과 예제는 pandas 2.3.3과 Matplotlib 공식 문서를 기준으로 작성했습니다.

- [pandas Chart visualization](https://pandas.pydata.org/pandas-docs/version/2.3.3/user_guide/visualization.html)
- [DataFrame.plot API](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.DataFrame.plot.html)
- [Series.plot API](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.Series.plot.html)
- [DataFrame.boxplot API](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.DataFrame.boxplot.html)
- [DataFrame.plot.hexbin API](https://pandas.pydata.org/pandas-docs/version/2.3.3/reference/api/pandas.DataFrame.plot.hexbin.html)
- [Matplotlib Introduction to Axes](https://matplotlib.org/stable/users/explain/axes/axes_intro.html)